In [ ]:
import torch
from torch_geometric.nn import HeteroDictLinear

# Cada tipo tem uma dimensionalidade de feature diferente
lin = HeteroDictLinear(
    in_channels={'user': 64, 'item': 32, 'tag': 16},
    out_channels=128,                 # saída uniforme
)

x_dict = {
    'user': torch.randn(10, 64),
    'item': torch.randn(50, 32),
    'tag':  torch.randn(20, 16),
}

out = lin(x_dict)
print(out['user'].shape)   # torch.Size([10, 128])
print(out['item'].shape)   # torch.Size([50, 128])
print(out['tag'].shape)    # torch.Size([20, 128])

torch.Size([10, 128])
torch.Size([50, 128])
torch.Size([20, 128])


In [2]:
import sys
import torch
from pathlib import Path
from typing import List

root_dir = Path("/home/milnner00/projects/gnn_recsys/code")
sys.path.append(str(root_dir))

from recsys.splits import per_src_leave_k_edges, per_src_k_fold
from recsys.datasets import ML32MGraph
from recsys.data import rebuild_hetero
from torch_geometric.loader import LinkNeighborLoader, NeighborLoader
from torch_geometric.data import HeteroData
import torch.nn.functional as F

In [6]:
root = "/home/milnner00/projects/gnn_recsys/data/ml-32m"
g = ML32MGraph(root=root, device="cpu", load_edge_ts=True)
train_idx, test_idx = per_src_leave_k_edges(g.edge_index,g.num_users, k=1, seed=42)
hetero_data = g.to_hetero()
print(hetero_data["movie"].x.size())
edge = ("user", "rates", "movie")
rev_edge = ("movie", "rated_by", "user")


torch.Size([87585, 3072])


In [ ]:
graph.to_hetero()[('user', 'rates', 'movie')].edge_label


tensor([4.0000, 1.0000, 2.0000,  ..., 0.5000, 3.5000, 4.5000])

In [ ]:
import copy
import statistics as stats
import torch

from recsys import train
from recsys.aggr import aggregate_neighbors
from recsys.config import Config
from recsys.eval import evaluate_full_graph
from recsys.models.hetero_sage import HeteroGraphSAGE

# ===================== dados =====================
seed      = 42
leave_k   = 1
batch_size = 4096

root = "/home/milnner00/projects/gnn_recsys/data/ml-32m"
g    = ML32MGraph(root=root, device="cpu", load_edge_ts=True)

train_idx, test_idx = per_src_leave_k_edges(
  g.edge_index, g.num_users, k=leave_k, seed=seed)

hetero_data = g.to_hetero()
edge     = ("user", "rates", "movie")
rev_edge = ("movie", "rated_by", "user")

hetero_data["movie"].x = hetero_data["movie"].x[:, :1024]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_edge_index = hetero_data[edge].edge_index[:, train_idx]

fold_ids, folds_local = per_src_k_fold(
  train_edge_index, g.num_users, n_folds=5,
  seed=seed)

folds = [
  (train_idx[tr_local], train_idx[va_local])
  for tr_local, va_local in folds_local
]

in_channels_dict = {"user": 1024, "movie": 1024}   # ajuste se agregar diferente

base_cfg = Config(
  ratio=5,
  negative_sampling="perm",
  filter_false_negatives=True,
  bpr_reduction="mean_positives",
  patience=10, patience_metric="ndcg@{k}", patience_mode="max",
  k=10, num_epochs=10_000, l2_reg=1e-4, lr=1e-3,
  eval_mode="full_graph",
  num_neighbors=[15, 10],
  eval_num_neighbors=[15, 10],
  eval_num_negatives=100,
  n_users=g.num_users, n_items=g.num_items,
)

# ===================== CV =====================
fold_results = []

for f, (tr_global, va_global) in enumerate(folds):
    print(f"\n=== Fold {f} | train={tr_global.numel()} | val={va_global.numel()}")

    # 1) reconstrói grafo do fold SÓ com tr_global
    fold_data = rebuild_hetero(hetero_data, tr_global, edge, rev_edge)

    # 2) user features agregadas SÓ com tr_global (sem leakage de va_global)
    fold_data["user"].x = aggregate_neighbors(
      tr_global, fold_data["movie"].x,
    )

    # 3) modelo e opt NOVOS por fold
    model = HeteroGraphSAGE(
      fold_data.metadata(),
      in_channels_dict = in_channels_dict,
      hidden_channels  = 64,
      num_layers       = 2,
      dropout          = 0.1,
      aggr             = "mean",
      normalize        = True,
      input_proj       = False,
    ).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=base_cfg.lr)

    # 4) RNG próprio por fold
    rgn = torch.Generator(device=device).manual_seed(seed * 1000 + f)
    cfg = copy.replace(base_cfg, rgn=rgn) if hasattr(copy, "replace") else base_cfg

    # 5) loaders
    train_loader = LinkNeighborLoader(
      fold_data, num_neighbors=cfg.num_neighbors, batch_size=batch_size,
      edge_label_index=(edge, fold_data[edge].edge_index),
      edge_label       =(edge, fold_data[edge].edge_label),
      shuffle=True, num_workers=8,
    )
    val_loader = LinkNeighborLoader(
      fold_data, num_neighbors=cfg.num_neighbors, batch_size=batch_size,
      edge_label_index=(edge, hetero_data[edge].edge_index[:, va_global]),
      edge_label       =(edge, hetero_data[edge].edge_label[:, va_global]),
      shuffle=False, num_workers=8,
    )

    # 6) treino
    out = train(
        train_loader, val_loader, model, opt, device, edge,
        train_edge_index = fold_data[edge].edge_index,   # só tr_global
        cfg              = cfg,
        full_data        = fold_data,                     # popularidade = tr_global
        eval_edge_index  = hetero_data[edge].edge_index[:, va_global],
    )
    fold_results.append(out)

# ===================== sumário CV =====================
keys = [k for k in fold_results[0]["history"][0]
        if k not in ("epoch", "loss", "fn_rate")]
for k in keys:
    vals = [r["history"][-1].get(k) for r in fold_results]
    vals = [v for v in vals if v is not None]
    if vals:
        print(f"{k}: {stats.mean(vals):.4f} ± {stats.stdev(vals) if len(vals)>1 else 0:.4f}")

# ===================== treino final + test =====================
print("\n=== Treino final em train_idx completo ===")

# features de user agregadas com o conjunto de treino final (sem test_idx)
final_data = rebuild_hetero(hetero_data, train_idx, edge, rev_edge)
final_data["user"].x = aggregate_neighbors(train_idx, final_data["movie"].x)

final_model = HeteroGraphSAGE(
    final_data.metadata(),
    in_channels_dict = in_channels_dict,
    hidden_channels  = 64, num_layers = 2, dropout = 0.1,
    aggr = "mean", normalize = True, input_proj = False,
).to(device)
final_opt = torch.optim.Adam(final_model.parameters(), lr=base_cfg.lr)

final_train_loader = LinkNeighborLoader(
    final_data, num_neighbors=base_cfg.num_neighbors, batch_size=batch_size,
    edge_label_index=(edge, final_data[edge].edge_index),
    edge_label       =(edge, final_data[edge].edge_label),
    shuffle=True, num_workers=8,
)
# loader de val dummy (early stopping interno em train usa métrica)
dummy_val = final_train_loader

final_out = train(
    final_train_loader, dummy_val, final_model, final_opt, device, edge,
    train_edge_index = final_data[edge].edge_index,
    cfg              = base_cfg,
    full_data        = final_data,
    eval_edge_index  = final_data[edge].edge_index,  # val interno = train
)

test_metrics = evaluate_full_graph(
    final_model, final_data, "user", "movie", edge,
    device=device, k=base_cfg.k,
    num_negatives=base_cfg.eval_num_negatives,
    eval_edge_index=hetero_data[edge].edge_index[:, test_idx],
    seed=base_cfg.eval_seed,
)
print("TESTE:", test_metrics)

SyntaxError: invalid syntax (945115626.py, line 531)

In [ ]:
n_folds = 5
src = torch.tensor([0, 0, 0, 1, 1, 1, 2, 3, 3, 4], dtype=torch.long)
E = src.numel()
rand = torch.rand(E)

print("src: ", src)

by_rand = torch.argsort(rand)
# valor permutado
print("src[by_rand]: ", src[by_rand])

# dada a permutação retorne os indexes para que fique ordenado
argsort_by_rand = torch.argsort(src[by_rand], stable=True)
print("argsort: ",argsort_by_rand)
# ordenado novamente
print("src[by_rand][argsort]: ", src[by_rand][argsort_by_rand])
print()

# idx em by_rand ordenados para retornar src[idx] em ordem crescente
order = by_rand[argsort_by_rand]


print("order = by_rand[argsort]: ", order)
print()

order_src = src[order]
# ordenado depois de desordenar
print("order_src: ", order_src)
counts = torch.bincount(order_src, minlength=3)

offsets = torch.cat([torch.zeros(1,dtype=torch.long), counts.cumsum(0)])

print()
# print("deg: ", offsets[1:] - offsets[:-1])

# print("arange: ",torch.arange(6, dtype=torch.long))
# print()


print("offsets: ",offsets)
print("offset[order_src]: ", offsets[order_src])
rank = torch.arange(E, dtype=torch.long) - offsets[order_src]
print("rank: ", rank)
print(rank % 3)

src:  tensor([0, 0, 0, 1, 1, 1, 2, 3, 3, 4])
src[by_rand]:  tensor([0, 1, 4, 0, 2, 1, 1, 3, 3, 0])
argsort:  tensor([0, 3, 9, 1, 5, 6, 4, 7, 8, 2])
src[by_rand][argsort]:  tensor([0, 0, 0, 1, 1, 1, 2, 3, 3, 4])

order = by_rand[argsort]:  tensor([1, 2, 0, 5, 3, 4, 6, 8, 7, 9])

order_src:  tensor([0, 0, 0, 1, 1, 1, 2, 3, 3, 4])

offsets:  tensor([ 0,  3,  6,  7,  9, 10])
offset[order_src]:  tensor([0, 0, 0, 3, 3, 3, 6, 7, 7, 9])
rank:  tensor([0, 1, 2, 0, 1, 2, 0, 0, 1, 0])
tensor([0, 1, 2, 0, 1, 2, 0, 0, 1, 0])


In [ ]:
# 4. Round-robin: fold = rank % n_folds
fold_local = (rank % n_folds).to(torch.long)

# 5. Exclui sources com count <= 1 (não podem ser val)
fold_local[counts[order_src] <= 1] = -1

# 6. Volta à ordem original
fold_ids = torch.empty(E, dtype=torch.long)
fold_ids[order] = fold_local

# 7. Deriva (train, val) como complemento
all_idx = torch.arange(E)
folds = []
for f in range(n_folds):
    val_mask  = fold_ids == f
    tr_mask   = ~val_mask
    folds.append((all_idx[tr_mask], all_idx[val_mask]))

In [ ]:
folds

[(tensor([0, 2, 3, 4, 6, 7, 9]), tensor([1, 5, 8])),
 (tensor([0, 1, 4, 5, 6, 8, 9]), tensor([2, 3, 7])),
 (tensor([1, 2, 3, 5, 6, 7, 8, 9]), tensor([0, 4])),
 (tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), tensor([], dtype=torch.int64)),
 (tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), tensor([], dtype=torch.int64))]

In [ ]:
fold_ids

tensor([ 2,  0,  1,  1,  2,  0, -1,  1,  0, -1])